In [1]:
import numpy as np
import os

target_folder = '/data6/Users/yeonjoon/VcbMVAStudy/TabNet_template/TabNET_model/largePhaseSpace_B_MultiClass_EarlyStopLoss'
arr = np.load(f'{target_folder}/data.npz', allow_pickle=True)

In [2]:
info = np.load(f'{target_folder}/fold0/info.npy', allow_pickle=True)

In [3]:
varlist = info[()]['data_info']['varlist']

In [4]:
arr.keys()

KeysView(NpzFile '/data6/Users/yeonjoon/VcbMVAStudy/TabNet_template/TabNET_model/largePhaseSpace_B_MultiClass_EarlyStopLoss/data.npz' with keys: X, y, weight, folds_lab, n_splits...)

In [13]:
np.unique(arr['folds_lab'], return_counts=True)

(array([0, 1, 2], dtype=int16), array([3542464, 3543350, 3543150]))

In [5]:
train_features = arr['X']
train_y = arr['y']
train_weight = arr['weight']


In [14]:
def add_quantile_axis(ax, x_visible, *, ticks_pct=None, draw_guides=True, eps=1e-9):
    """
    ax         : 메인 x축이 있는 Axes
    x_visible  : 현재 플롯에 사용한 x 데이터(마스크/범위 적용 후)  ← unweighted, count-based
    ticks_pct  : 예) [0, 90, 95, 99, 99.9, 100]
    """
    if ticks_pct is None:
        ticks_pct = np.array([0, 90, 95, 99, 99.9, 100.0], dtype=float)

    xv = np.asarray(x_visible)
    xv = xv[np.isfinite(xv)]
    if xv.size < 2:
        return None

    # 경험적 CDF (count-based)
    xs = np.sort(xv)
    p  = np.linspace(0.0, 1.0, xs.size)

    # 올바른 변환쌍: forward=x→q, inverse=q→x
    def x_to_q(xx):
        xx = np.asarray(xx)
        return np.interp(xx, xs, p)

    def q_to_x(q):
        q = np.asarray(q)
        return np.interp(q, p, xs)

    # ⬅ 순서 교정!
    secax = ax.secondary_xaxis('top', functions=(x_to_q, q_to_x))
    secax.set_xlabel("Quantile (count-based)")

    # secax 만든 뒤에 추가
    secax.tick_params(axis='x', direction='in', length=6, pad=-10)  # pad를 음수로 ↓ 안쪽으로
    # 필요하면 톱 스파인 살짝 아래로 내리기 (플롯 내부로 2%):
    secax.spines['top'].set_position(('axes', 0.98))

    # 현재 xlim에 해당하는 q-범위만 표시
    lo, hi = ax.get_xlim()
    qlo, qhi = x_to_q([lo, hi])
    qmin, qmax = (min(qlo, qhi), max(qlo, qhi))

    ticks_pct = np.asarray(ticks_pct, dtype=float)
    qticks = ticks_pct / 100.0
    m = (qticks >= qmin - eps) & (qticks <= qmax + eps)
    qt = qticks[m]

    secax.set_xticks(qt)
    # 99.9 같은 소수 라벨 보존
    secax.set_xticklabels([f"{t:g}%" for t in ticks_pct[m]])

    # 가이드라인(옵션)
    if draw_guides and qt.size:
        y0, y1 = ax.get_ylim()
        ax.vlines(q_to_x(qt), y0, y1, linewidth=0.5, alpha=0.7, colors='salmon', linestyles='dotted')

    return secax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import mplhep as hep

# =========================
# 사용자 설정
# =========================
hep.style.use("CMS")           # 스타일: "CMS" / "ATLAS" / "LHCb" 등
YEAR_LABEL = "Run2"            # CMS 라벨 year 표기 (예: "2016–2018", "2018", "Run2")
LUMI_STR   = None              # 예: '137 fb$^{-1}$ (13 TeV)' 또는 None
OUT_PDF    = "label_distributions_by_feature.pdf"
OUT_PDF = os.path.join(target_folder, OUT_PDF)

NUM_BINS   = 50
LOW_Q      = 0.0               # x-축 하/상위 퍼센타일 컷
HIGH_Q     = 100.0

# 필수 입력:
# - train_features: shape (N, D)
# - train_y:        shape (N,)
# - (선택) train_weight: shape (N,)
try:
    train_weight
except NameError:
    train_weight = None  # 없으면 None

# varlist가 없다면 자동 대체(피처 이름)
try:
    varlist
    _HAS_VARLIST = True
except NameError:
    _HAS_VARLIST = False

classes = np.unique(train_y)
N, D = train_features.shape

for fold in np.unique(arr['folds_lab']):
    fold_mask = (arr['folds_lab'] == fold)
    with PdfPages(OUT_PDF.replace('.pdf', f'_fold{fold}.pdf')) as pdf:
        for j in range(D):
            x_all = train_features[:, j]

            # 유효값 마스크
            m = np.isfinite(x_all)
            if train_weight is not None:
                m &= np.isfinite(train_weight)
                
            m &= fold_mask  # 현재 폴드

            x = x_all[m]
            y = train_y[m]
            w = (train_weight[m] if train_weight is not None else None)

            # 데이터 없으면 건너뜀
            if x.size == 0:
                continue

            # 퍼센타일 기반 범위
            lo, hi = np.percentile(x, [LOW_Q, HIGH_Q])
            if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
                lo, hi = np.nanmin(x), np.nanmax(x)
                if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
                    lo, hi = float(np.min(x)) - 0.5, float(np.max(x)) + 0.5

            # 피처 이름
            if _HAS_VARLIST and isinstance(varlist, (list, tuple)) and len(varlist) == D:
                feat_name = str(varlist[j])
            else:
                feat_name = f"feature[{j}]"

            # Figure/Axes 생성
            fig, ax = plt.subplots(figsize=(12, 10))

            # 클래스별 히스토그램 (HEP 스타일엔 step이 깔끔)
            for c in classes:
                xm = x[y == c]
                if xm.size == 0:
                    continue
                wm = (w[y == c] if w is not None else None)
                ax.hist(
                    xm,
                    bins=NUM_BINS,
                    range=(lo, hi),
                    histtype="step",
                    density=True,
                    weights=wm,
                    label=f"class {c} (n={xm.size})",
                )
                
            add_quantile_axis(ax, x)

            # CMS 라벨(축 안쪽 좌상단)
            try:
                hep.cms.label(ax=ax, data=False, year=YEAR_LABEL, lumi=LUMI_STR)
            except Exception:
                # mplhep 버전 호환을 위한 안전장치
                hep.cms.text("Preliminary", ax=ax)

            # ---- 제목: 축 바깥 최상단 중앙 (겹침 X) ----
            fig.text(
                0.5, 0.99,
                f"{feat_name} — label-wise distribution",
                ha="center", va="top"
            )

            # 축/범례/그리드
            ax.set_xlabel(feat_name)
            ax.set_ylabel("Density")
            ax.grid(alpha=0.3)
            ax.legend(frameon=False, loc="best")

            # 제목 공간 확보
            plt.tight_layout(rect=(0, 0, 1, 0.96))
            pdf.savefig(fig)
            plt.close(fig)

    print(f"Saved: {OUT_PDF.replace('.pdf', f'_fold{fold}.pdf')}")

Saved: /data6/Users/yeonjoon/VcbMVAStudy/TabNet_template/TabNET_model/largePhaseSpace_B_MultiClass_EarlyStopLoss/label_distributions_by_feature_fold0.pdf
Saved: /data6/Users/yeonjoon/VcbMVAStudy/TabNet_template/TabNET_model/largePhaseSpace_B_MultiClass_EarlyStopLoss/label_distributions_by_feature_fold1.pdf
